# Data Collection and Ingestion

In any data-driven project, the first critical step is acquiring relevant and reliable data from multiple sources. In this project, the objective is to build a predictive model for bike-sharing demand. To achieve this, we must gather both historical and real-time data from diverse sources.

This notebook focuses on constructing the data ingestion pipeline by collecting data from:

- Web sources (Wikipedia pages containing global bike-sharing systems)
- APIs (OpenWeather API for weather forecasts)
- Structured datasets (CSV files containing historical bike-sharing data)

Each of these sources provides complementary information:
- Web scraping provides system-level metadata
- APIs provide dynamic environmental conditions
- CSV datasets provide historical observations

By combining these, we create a comprehensive dataset suitable for downstream analysis and modeling.

## Import Required Libraries

To perform data collection tasks, we need several libraries:

- `requests`: to send HTTP requests
- `BeautifulSoup`: to parse HTML content for web scraping
- `pandas`: to structure and manipulate tabular data
- `json`: to handle API responses

In [2]:
# Import libraries for data collection and processing

import requests                     # For making HTTP requests to APIs and web pages
from bs4 import BeautifulSoup       # For parsing HTML content (web scraping)
import pandas as pd                 # For data manipulation and DataFrame creation
import json                         # For handling JSON data from APIs

## Web Scraping: Global Bike Sharing Systems

The first data source is a Wikipedia page listing bike-sharing systems around the world. This provides metadata such as:

- System name
- Location
- Country
- Operational details

We will extract tabular data from this page using HTML parsing techniques.

In [3]:
# Import required libraries
import requests  # For HTTP requests
from bs4 import BeautifulSoup  # For HTML parsing
import pandas as pd  # For data handling
from io import StringIO  # ✅ FIX: Forces pandas to treat input as HTML


# Define URL
url = "https://en.wikipedia.org/wiki/List_of_bicycle-sharing_systems"


# Add headers
headers = {
    "User-Agent": "Mozilla/5.0"
}


# Send request
response = requests.get(url, headers=headers)


# Check response
print("Status Code:", response.status_code)


# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")


# Extract all tables
tables = soup.find_all("table")


# Debug count
print("Number of tables found:", len(tables))


# Store parsed tables
df_list = []


# Loop through tables
for i, table in enumerate(tables):
    
    try:
        # ✅ Convert HTML table to string
        table_html = str(table)
        
        # ✅ Wrap in StringIO to prevent pandas from treating it as file path
        df = pd.read_html(StringIO(table_html))[0]
        
        # Append to list
        df_list.append(df)
        
        print(f"Table {i} parsed successfully")
    
    except Exception as e:
        print(f"Table {i} failed: {e}")


# Check total parsed tables
print("\nTotal successfully parsed tables:", len(df_list))


# Select main table (usually index 0 for Wikipedia)
bike_sharing_df = df_list[0]


# Preview
print("\nPreview:")
print(bike_sharing_df.head())

Status Code: 200
Number of tables found: 4
Table 0 parsed successfully
Table 1 parsed successfully
Table 2 parsed successfully
Table 3 parsed successfully

Total successfully parsed tables: 4

Preview:
     Country  Country.1          City / Region                 Name  \
0    Albania    Albania              Tirana[5]             Ecovolis   
1  Argentina  Argentina     Buenos Aires[6][7]              Ecobici   
2  Argentina  Argentina            Mendoza[10]            Metrobici   
3  Argentina  Argentina                Rosario  Mi Bici Tu Bici[11]   
4  Argentina  Argentina  San Lorenzo, Santa Fe             Biciudad   

              System                      Operator          Launched  \
0                NaN                           NaN        March 2011   
1  Serttel Brasil[8]  Bike In Baires Consortium[9]              2010   
2                NaN                           NaN              2014   
3                NaN                           NaN   2 December 2015   
4          

## Data Cleaning: Bike Sharing Systems

After extracting the table, we perform basic cleaning:

- Remove unnecessary columns
- Rename columns for clarity
- Handle missing values if needed

This ensures the dataset is structured properly for later use.

In [4]:
# Select relevant columns (example — adjust based on actual table structure)

bike_sharing_df = bike_sharing_df.iloc[:, :5]

# Rename columns for clarity
bike_sharing_df.columns = ['System', 'Location', 'Country', 'Year', 'Bikes']

# Display cleaned dataset
bike_sharing_df.head()

,System,Location,Country,Year,Bikes
0,Albania,Albania,Tirana[5],Ecovolis,NaN
1,Argentina,Argentina,Buenos Aires[6][7],Ecobici,Serttel Brasil[8]
2,Argentina,Argentina,Mendoza[10],Metrobici,NaN
3,Argentina,Argentina,Rosario,Mi Bici Tu Bici[11],NaN
4,Argentina,Argentina,"San Lorenzo, Santa Fe",Biciudad,Biciudad


## API Data Collection: Weather Forecast

Weather conditions play a significant role in bike-sharing demand. To incorporate environmental factors, we retrieve weather forecast data using the OpenWeather API.

This API provides:

- Temperature
- Humidity
- Wind speed
- Weather conditions
- Forecast timestamps

These variables will later be used as predictors in the regression model.

In [5]:
# Define API endpoint and parameters

api_key = " "   # Replace with your OpenWeather API key
city = "Seoul"

url = "https://api.openweathermap.org/data/2.5/forecast"

params = {
    "q": city,
    "appid": api_key,
    "units": "metric"
}

# Send request to API
response = requests.get(url, params=params)

# Convert response to JSON
weather_data = response.json()

# Inspect structure
weather_data.keys()

dict_keys(['cod', 'message', 'cnt', 'list', 'city'])

## Extract Relevant Weather Features

The API returns nested JSON data. We extract relevant attributes such as:

- Temperature
- Humidity
- Wind speed
- Timestamp

These will be converted into a structured DataFrame.

In [6]:
# Initialize empty list to store extracted data
weather_records = []

# Loop through forecast entries
for item in weather_data['list']:
    
    record = {
        "datetime": item['dt_txt'],
        "temperature": item['main']['temp'],
        "humidity": item['main']['humidity'],
        "wind_speed": item['wind']['speed'],
        "weather": item['weather'][0]['main']
    }
    
    weather_records.append(record)

# Convert to DataFrame
weather_df = pd.DataFrame(weather_records)

# Display sample data
weather_df.head()

,datetime,temperature,humidity,wind_speed,weather
0,2026-04-22 15:00:00,12.66,52,0.74,Clouds
1,2026-04-22 18:00:00,13.03,48,1.48,Clouds
2,2026-04-22 21:00:00,12.44,45,1.71,Clouds
3,2026-04-23 00:00:00,14.82,31,3.16,Clouds
4,2026-04-23 03:00:00,19.11,18,3.86,Clouds


## Summary

In this notebook, we established the data ingestion layer by:

- Extracting structured data from a web page using web scraping
- Collecting real-time weather data using an API
- Converting raw data into structured DataFrames

These datasets form the foundation for subsequent stages, including data wrangling, exploratory analysis, and predictive modeling.

---

## Author & Acknowledgment

**Author:**  
Deepan Mehta  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook is part of a comprehensive data analytics capstone project focused on predicting bike-sharing demand using environmental and temporal variables.

The structure and foundational methodology are inspired by instructional materials from the IBM Skills Network capstone labs.

Special acknowledgment is given to the original contributors of the lab materials, including:

- Yan Luo  
- Jeff Grossman  
- Rav Ahuja  

These materials provided guidance on data collection, exploratory analysis, and modeling techniques, which have been further extended and adapted in this project.

---

## Project Context

This notebook contributes to a larger end-to-end data science pipeline, including:

- Data Collection (Web scraping, APIs, datasets)
- Data Wrangling and Transformation (ETL)
- Exploratory Data Analysis (EDA)
- Predictive Modeling (Regression & Regularization)
- Interactive Dashboard Development (R Shiny)

---

## Notes

All code, structure, and explanations in this notebook have been independently rewritten and enhanced to reflect a production-oriented workflow suitable for real-world data science applications.

The repository link will be updated with the finalized project structure, including notebooks, reports, and deployment components.

---